# 14 · Data Engineering for ML — Feeding the Loop

**Part 2 · Machine Learning Systems** · *Prerequisites: 03, 10–11* · *Book mapping: DMLS ch. 3*

Models eat data that somebody had to move, clean, type, and lay out. This notebook is that
somebody. It is also where Part 1 pays its first direct dividend: the formats question is
nb 03's row-vs-column trade industrialized, the pipeline is nb 10's batch factory, and the
fresh path is nb 11's log.

**Learning objectives.**

1. Build an **ETL pipeline**: messy JSON event logs → validated, typed, **date-partitioned
   Parquet** — with loud accounting for every dropped record.
2. Benchmark CSV vs JSON vs Parquet on size and query speed; inspect Parquet's internals
   (row groups, column statistics) to see *why* it wins.
3. Use partition + column **pruning**: read only what the query mentions.
4. Build a **streaming micro-batch ingester** that routes late events to the right
   event-date partition (nb 11's event time, applied to storage).
5. Place warehouse / lake / lakehouse and ETL vs ELT on one map.

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as pads
import pyarrow.parquet as pq

from sysdes import scratch_dir, viz

viz.use_style()
rng = np.random.default_rng(seed=14)
LAKE = scratch_dir("14-data-engineering/lake")
RAW = scratch_dir("14-data-engineering/raw")

## 1. Extract, Transform, Load — with receipts

Raw reality: applications emit JSON event lines (nb 02's schema-on-read documents — every
producer slightly different, some lines simply broken). We synthesize 8 days of clickstream
with a deliberate ~1% of garbage lines, because your real logs have them too:

In [ ]:
DAYS = [f"2026-08-{d:02d}" for d in range(1, 9)]
EVENT_TYPES = ["view", "click", "add_to_cart", "purchase"]

lines, n_garbage = [], 0
for i in range(200_000):
    day = DAYS[int(rng.integers(len(DAYS)))]
    event = {
        "ts": f"{day}T{int(rng.integers(24)):02d}:{int(rng.integers(60)):02d}:00",
        "user_id": int(rng.integers(20_000)),
        "event_type": EVENT_TYPES[int(rng.integers(4))],
        "amount": round(float(rng.exponential(30)), 2),
    }
    if rng.random() < 0.01:                       # the inevitable corruption
        lines.append('{"ts": "2026-08-0')         # truncated mid-write
        n_garbage += 1
    else:
        lines.append(json.dumps(event))

raw_path = RAW / "events.jsonl"
raw_path.write_text("\n".join(lines))
print(f"raw log: {len(lines):,} lines ({n_garbage} corrupt), "
      f"{raw_path.stat().st_size/1e6:.1f} MB")

The pipeline. House rules from the global course conventions apply doubly here: **nothing
fails silently** — every dropped record is counted and reported, because "the pipeline ran
green but 30% of rows quietly vanished" is the classic way training data goes bad
(nb 15/16 will feel any sin committed here):

In [ ]:
def extract_transform(path) -> tuple[pd.DataFrame, dict]:
    """Parse, validate, type. Returns (clean dataframe, audit counters)."""
    rows, audit = [], {"read": 0, "malformed_json": 0, "failed_validation": 0}
    for line in path.read_text().splitlines():
        audit["read"] += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            audit["malformed_json"] += 1
            continue
        # Validation gate: required fields, sane types/ranges. Reject > repair.
        if not (isinstance(rec.get("user_id"), int)
                and rec.get("event_type") in EVENT_TYPES
                and isinstance(rec.get("amount"), (int, float))
                and rec["amount"] >= 0):
            audit["failed_validation"] += 1
            continue
        rows.append(rec)

    df = pd.DataFrame(rows)
    df["ts"] = pd.to_datetime(df["ts"])
    df["event_date"] = df["ts"].dt.date.astype(str)     # the partition key
    df["event_type"] = df["event_type"].astype("category")  # dictionary encoding, nb 03
    df["amount"] = df["amount"].astype("float32")
    return df, audit

events, audit = extract_transform(raw_path)
print(f"audit: {audit} -> kept {len(events):,}")
assert audit["malformed_json"] == n_garbage, "every corrupt line accounted for"
assert audit["read"] == audit["malformed_json"] + audit["failed_validation"] + len(events), \
    "conservation law: read = kept + rejected (nothing silently vanished)"

In [ ]:
def load_partitioned(df: pd.DataFrame, base, part_name: str = "part-000") -> None:
    """LOAD: write hive-style date partitions - a directory per day.
    Partitioning by the query-time filter column is nb 06's key-range
    partitioning, reborn as a directory layout."""
    for day, group in df.groupby("event_date", observed=True):
        part_dir = base / f"event_date={day}"
        part_dir.mkdir(parents=True, exist_ok=True)
        group.drop(columns=["event_date"]).to_parquet(part_dir / f"{part_name}.parquet")

load_partitioned(events, LAKE)
n_files = len(list(LAKE.rglob("*.parquet")))
print(f"lake layout: {n_files} partition files under {LAKE.name}/event_date=YYYY-MM-DD/")
assert n_files == len(DAYS)

## 2. Why Parquet — measured, then dissected

Write the *same* table three ways and put every claim on a scale. Parquet is columnar
(nb 03 §5 industrialized): per-column layout, dictionary + run-length compression, and
per-chunk statistics.

In [ ]:
flat_csv = RAW / "events.csv"
flat_jsonl = RAW / "events_clean.jsonl"
flat_parquet = RAW / "events.parquet"
events.to_csv(flat_csv, index=False)
events.to_json(flat_jsonl, orient="records", lines=True)
events.to_parquet(flat_parquet)

sizes = {p.name: p.stat().st_size / 1e6 for p in (flat_jsonl, flat_csv, flat_parquet)}

t0 = time.perf_counter(); pd.read_csv(flat_csv); t_csv_full = time.perf_counter() - t0
t0 = time.perf_counter(); pd.read_parquet(flat_parquet); t_pq_full = time.perf_counter() - t0
t0 = time.perf_counter()
pd.read_parquet(flat_parquet, columns=["amount"])       # ONE column, not the file
t_pq_col = time.perf_counter() - t0

print({k: f"{v:.1f} MB" for k, v in sizes.items()})
print(f"full read - csv: {t_csv_full*1e3:.0f} ms, parquet: {t_pq_full*1e3:.0f} ms; "
      f"single column via parquet: {t_pq_col*1e3:.0f} ms")
assert sizes["events.parquet"] < sizes["events.csv"] < sizes["events_clean.jsonl"], \
    "text formats repeat names & digits; parquet compresses columns"
assert t_pq_col < t_pq_full, "columnar: reading one column skips the others' bytes"

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(sizes.keys(), sizes.values(), color=["#9e9e9e", "#ef6c00", "#2e7d32"])
ax.set_ylabel("file size (MB)")
ax.set_title("One table, three formats")
plt.show()

Crack a file open — the metadata *is* the query accelerator:

In [ ]:
meta = pq.ParquetFile(flat_parquet).metadata
stats = meta.row_group(0).column(events.columns.get_loc("amount")).statistics
print(f"row groups: {meta.num_row_groups}, rows: {meta.num_rows:,}")
print(f"'amount' column chunk stats: min={stats.min:.2f} max={stats.max:.2f}")
assert stats.min >= 0 and stats.max <= float(events.amount.max()) + 0.01

# Those min/max stats mean an engine can SKIP whole chunks/files whose range
# can't match a filter - nb 03's "sort order is a pruning index" in the wild.

And the partition layout is the coarsest, cheapest index of all. A training query — "August
2nd and 3rd, purchases only, user and amount columns" — touches 2 of 8 directories and 3 of
5 columns:

In [ ]:
dataset = pads.dataset(LAKE, partitioning="hive")
t0 = time.perf_counter()
subset = dataset.to_table(
    columns=["user_id", "amount"],
    filter=(pads.field("event_date").isin(["2026-08-02", "2026-08-03"])
            & (pads.field("event_type") == "purchase")),
).to_pandas()
t_pruned = time.perf_counter() - t0

truth = events[(events.event_date.isin(["2026-08-02", "2026-08-03"]))
               & (events.event_type == "purchase")]
assert len(subset) == len(truth), "pruned read returns exactly the right rows"
print(f"pruned read: {len(subset):,} rows in {t_pruned*1e3:.0f} ms "
      f"(scanned 2/8 partitions, 2/5 columns) - this is what makes lakes queryable")

## 3. The map: warehouse, lake, lakehouse; ETL vs ELT

- **Data warehouse**: schema-on-write (nb 02), curated tables, SQL engine attached —
  expensive, trustworthy, great for BI.
- **Data lake**: cheap object storage full of files (what we just built) — schema-on-read,
  anything lands, quality varies by directory. ML training loves it (models want raw-ish
  history); analysts drown without conventions.
- **Lakehouse** (Delta/Iceberg/Hudi): the lake plus a *transactional table layer* —
  metadata that gives atomic multi-file commits, schema enforcement/evolution (nb 04!),
  and time travel. It is nb 07's atomicity retrofitted onto directories of Parquet — the
  cure for "the training job read the partition while it was half-written".
- **ETL vs ELT**: transform-then-land vs land-raw-then-transform-inside. ELT won broadly
  because storage got cheap and transformations iterate (nb 12's refold argument: keep raw
  history, re-derive at will). ETL survives where it must: PII redaction/consent filtering
  *before* data lands anywhere durable — some transformations are legal requirements, not
  conveniences.

## 4. The fresh path: streaming micro-batch ingestion

Nightly batch ETL means features are up to 24h stale. The streaming path tails the event
log (nb 11) and loads small batches continuously. The one design decision that matters:
partition by **event time**, not arrival time — late events (nb 11's outage stragglers)
must land in the partition where queries will look for them:

In [ ]:
class MicroBatchIngester:
    """Consumes event batches; appends one new part-file per touched partition.
    Each flush = new files (immutable, nb 03-style); nothing rewritten."""

    def __init__(self, base):
        self.base = base
        self.flushes = 0

    def ingest(self, batch: list[dict]) -> list[str]:
        df, _ = _frame(batch)
        self.flushes += 1
        touched = sorted(df.event_date.unique())
        load_partitioned(df, self.base, part_name=f"part-{self.flushes:03d}")
        return touched

def _frame(batch):
    df = pd.DataFrame(batch)
    df["ts"] = pd.to_datetime(df["ts"])
    df["event_date"] = df["ts"].dt.date.astype(str)
    return df, None

def count_for_day(day: str) -> int:
    d = pads.dataset(LAKE, partitioning="hive")
    return len(d.to_table(filter=pads.field("event_date") == day))

before = count_for_day("2026-08-08")

# Today (Aug 9) two batches stream in; one contains a LATE event from Aug 8
# (a phone that reconnected). Event-date partitioning routes it home:
ing = MicroBatchIngester(LAKE)
touched1 = ing.ingest([
    {"ts": "2026-08-09T10:00:00", "user_id": 7, "event_type": "view", "amount": 0.0},
    {"ts": "2026-08-09T10:00:05", "user_id": 8, "event_type": "click", "amount": 0.0},
])
touched2 = ing.ingest([
    {"ts": "2026-08-09T10:01:00", "user_id": 9, "event_type": "view", "amount": 0.0},
    {"ts": "2026-08-08T23:59:00", "user_id": 3, "event_type": "purchase", "amount": 9.5},
])

print(f"batch 1 touched partitions {touched1}; batch 2 touched {touched2}")
assert touched2 == ["2026-08-08", "2026-08-09"], "the late event went to YESTERDAY"
assert count_for_day("2026-08-08") == before + 1, \
    "queries for Aug 8 now include the straggler - event time, not arrival time"
assert count_for_day("2026-08-09") == 3
print("late event landed in its event-date partition; day-queries stay truthful")

The cost of this freshness is visible in the directory: every flush adds files — the
**small-files problem**. A partition accumulating thousands of tiny Parquet files makes
every read pay per-file overhead; the cure is a periodic **compaction job** that merges
small files into big ones. If that sounds exactly like nb 03's LSM compaction — it is,
at directory scale, and it's the same background debt to monitor (exercise 4).

## 5. Exercises

1. **Format duty roster.** Pick a format and one-line justification for: (a) 2 TB of
   training events queried by column and date; (b) a request payload between two services
   (nb 04's constraints apply); (c) debug logs a human greps at 3 a.m.; (d) a 200-row
   config table edited in pull requests.
2. **Partition the other way.** Training queries are "date range scans" (our layout wins).
   The serving path asks "this user's last 50 events" thousands of times per second. Why is
   the date-partitioned lake the wrong store for that query, and what does nb 03/06 say
   the right one looks like?
3. **ETL's last stand.** Your product logs may contain GDPR-protected fields. Argue
   precisely where redaction must happen relative to the lake, and what nb 12's
   crypto-shredding offers for data already landed.
4. **Small files.** After 30 days of 1-minute micro-batches, a partition has ~43k files.
   Sketch the compaction job (inputs, output, atomic swap — nb 10's build-artifact rule)
   and what must be true for readers running *during* compaction (lakehouse commit, nb 07).
5. **Effectively-once ingestion.** The ingester reads from an at-least-once broker
   (nb 11). A crash between "wrote part-file" and "committed offset" re-delivers the
   batch. Design idempotent loading using the batch's (partition, offset-range) as
   identity — where does the dedup record live so file-write and offset-commit can't
   diverge?

### Solutions

**1.** (a) Parquet, date-partitioned — today's whole notebook. (b) A schema'd binary
format (protobuf/Avro, nb 04) — compatibility matrix > compactness > human-readability.
(c) Plain text/JSONL — grep-ability *is* the requirement; compression via rotation.
(d) CSV/YAML in git — the "format" that matters is the diff and the review.

**2.** The lake orders bytes by (date, arrival) — one user's events are smeared across
every partition (nb 06 §1's exact skew argument, transposed): 50-event lookups would scan
gigabytes at request latency. The serving path wants a row store keyed by user_id (hash
partitioning + LSM/B-tree, nb 03/06) — which is precisely the **online/offline store
split** that nb 16 formalizes as a feature store. Same data, two layouts, derived from
one log (nb 12).

**3.** Redaction/consent filtering must run *before* durable landing (true ETL): once PII
is in the lake, every downstream copy, cache, and model checkpoint inherits the liability
(the end-to-end argument, in legal form). For history already landed: crypto-shredding —
per-user keys, delete the key on erasure request (nb 12 exercise 2) — plus re-derivation
of downstream views from the redacted source.

**4.** Nightly per partition: read all part-files, merge-sort (by ts), write K large
files to a *staging* path, then atomically swap directory pointers (lakehouse table
commit — the transactional metadata makes "which files constitute this table" an atomic
pointer flip, so in-flight readers finish on the old file list and new readers see the
new one). Delete old files only after the commit + a grace period. Inputs immutable,
output rebuilt, swap atomic: nb 10/12's rules verbatim.

**5.** Name the output deterministically by identity:
`part-{partition}-{first_offset}-{last_offset}.parquet`. On redelivery the ingester
checks existence (or overwrites byte-identically — same content, same name: naturally
idempotent) before committing offsets. The dedup "record" is thereby the file itself in
the same store as the data — one system, no cross-store race (nb 11 exercise 3's answer
(c), applied to files). A lakehouse commit log makes the check transactional.

## 6. Takeaways

- ETL with **receipts**: read = kept + rejected, always printed, always asserted. Silent
  row loss is how models rot upstream of any model code.
- **Parquet** = nb 03's column store industrialized: columnar layout + dictionary/RLE +
  chunk statistics. Partitions prune directories; columns prune bytes; stats prune chunks.
- Lake = cheap schema-on-read history (ELT, refold-friendly); warehouse = curated
  schema-on-write; lakehouse = lake + transactions (atomic multi-file commits).
- Stream ingestion: partition by **event time** so late data lands where queries look;
  pay the small-files tax and schedule compaction (LSM lessons at directory scale).
- The serving path needs the *same* data in a *different* layout — one log, many views
  (nb 12) — which is where feature stores (nb 16) come from.

**Further reading.** DMLS ch. 3; the Parquet format spec (row groups & statistics);
Armbrust et al., *Lakehouse* (CIDR 2021); Zaharia et al., *Delta Lake* (VLDB 2020).

**Next:** [15 · Training data](15-training-data.ipynb) — the data is in the lake; now
turn it into *labeled examples* without fooling yourself.